* The objective of this assignments is to build the **Decoder** part of the Transformer architecture.
* We will be using the **PyTorch** framework to implement the following components
  * Decoder Layer that contains
    * Multi-Head Masked Attention (MHMA) Module
    * Multi-Head Cross Attention (MHMA) Module
    * Position-wise Feed Forward Neural Network

  * Implement CLM

* **DO NOT** USE Built-in **TRANSFORMER LAYERS** as it affects the reproducibility.

* You will be given with a configuration file that contains information on various hyperparameters such as embedding dimension, vocabulary size,number heads and so on

* Use ReLU activation function and Stochastic Gradient Descent optimizer
* Here are a list of helpful Pytorch functions (does not mean you have to use all of them) for this subsequent assignments
  * [torch.matmul](https://pytorch.org/docs/stable/generated/torch.matmul.html#torch-matmul)
  * [torch.bmm](https://pytorch.org/docs/stable/generated/torch.bmm.html)
  * torch.swapdims
  * torch.unsqueeze
  * torch.squeeze
  * torch.argmax
  * [torch.Tensor.view](https://pytorch.org/docs/stable/generated/torch.Tensor.view.html)
  * [torch.nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
  * [torch.nn.Parameter](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html)
  * torch.nn.Linear
  * torch.nn.LayerNorm
  * torch.nn.ModuleList
  * torch.nn.Sequential
  * [torch.nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
  
* Important: Do not set any global seeds.

* Helpful resources to get started with

 * [Andrej Karpathys Nano GPT](https://github.com/karpathy/nanoGPT)
 * [PyTorch Source code of Transformer Layer](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html)



In [ ]:
import torch
from torch import Tensor

import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.nn.functional import one_hot

import torch.optim as optim

from  pprint import pprint
from yaml import safe_load
import copy
import requests
from io import BytesIO

In [ ]:
#do not edit this cell
config_url = "https://raw.githubusercontent.com/Arunprakash-A/LLM-from-scratch-PyTorch/main/config_files/dec_config.yml"
response = requests.get(config_url)
config = response.content.decode("utf-8")
config = safe_load(config)
pprint(config)

{'input': {'batch_size': 10, 'embed_dim': 32, 'seq_len': 8, 'vocab_size': 12},
 'model': {'d_ff': 128,
           'd_model': 32,
           'dk': 4,
           'dq': 4,
           'dv': 4,
           'n_heads': 8,
           'n_layers': 6}}


In [ ]:
vocab_size = config['input']['vocab_size']
batch_size = config['input']['batch_size']
seq_len = config['input']['seq_len']
embed_dim = config['input']['embed_dim']
dmodel = embed_dim
dq = torch.tensor(config['model']['dq'])
dk = torch.tensor(config['model']['dk'])
dv = torch.tensor(config['model']['dv'])
heads = torch.tensor(config['model']['n_heads'])
d_ff = config['model']['d_ff']

# Input tokens

* Generate a raw_input ids (without any special tokens appended to it)

* Since we will be using this as label after adding the special  \<start\> token, we use the variable name "label_ids"

* Keep the size of the `label_ids=(bs,seq_len-1)` as we insert a special token ids in the next step

In [ ]:
# do not edit this cell
data_url = 'https://github.com/Arunprakash-A/LLM-from-scratch-PyTorch/raw/main/config_files/w2_input_tokens'
r = requests.get(data_url)
label_ids = torch.load(BytesIO(r.content))

<ipython-input-4-8cc5628215f0>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  label_ids = torch.load(BytesIO(r.content))


* Let the first token_id be be a special `[start]` token (mapped to integer 0)
* If label_ids=$\begin{bmatrix}1&2\\3&4 \end{bmatrix}$, then we modify it as $\begin{bmatrix}0&1&2\\0&3&4 \end{bmatrix}$

In [ ]:
token_ids = torch.concatenate((torch.zeros(label_ids.shape[0]).reshape(-1,1), label_ids), dim = -1) # the first column of token_ids should be zeros and the rest of the columns come from label_ids
token_ids

tensor([[ 0.,  7.,  8.,  7.,  7.,  9.,  2.,  6.],
        [ 0., 10.,  1., 10.,  5.,  3.,  6.,  8.],
        [ 0.,  3.,  4.,  8.,  2., 10., 10., 10.],
        [ 0.,  4., 10.,  1.,  3.,  4.,  9.,  7.],
        [ 0.,  8.,  4.,  7.,  3.,  8., 10.,  5.],
        [ 0.,  9.,  1.,  8.,  5.,  9.,  9., 10.],
        [ 0.,  7.,  3.,  8.,  2.,  5.,  1.,  5.],
        [ 0.,  3.,  3.,  2.,  1.,  4.,  1.,  1.],
        [ 0., 10.,  9.,  9.,  9.,  6.,  9.,  2.],
        [ 0.,  3.,  6.,  6.,  3.,  5.,  4.,  5.]])

# Implement the following components of a decoder layer

 * Multi-head Masked Attention (MHMA)
 * Multi-head Cross Attention (MHCA)
 * Postion-wise FFN

* Randomly initialize the parameters using normal distribution with the following seed values
  * $W_Q:$(seed=43)
  * $W_K:$(seed=44)
  * $W_V:$(seed=45)
  * $W_O:$(seed=46)

* Remember that, Multi-head cross atention takes two represnetation. One is the encoder output and the other one is the output from masked attetnion sub-layer.

* However, in this assignment, we will fix it to a random matrix.

In [ ]:
class MHCA(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHCA,self).__init__()
    self.Wq = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(43)))
    self.Wk = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(44)))
    self.Wv = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(45)))
    self.Wo = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(46)))
    self.heads = heads
    self.dq = dq
  # your method definitions go here (if you want to)

  def split_heads(self, x):
    batch_size, seq_len, embed_dim = x.size()
    return x.view(batch_size, seq_len, self.heads, self.dq).transpose(1,2)

  def combine_heads(self, x):
    batch_size, heads, seq_len, attn_len = x.size()
    return x.transpose(1,2).contiguous().view(batch_size, seq_len, heads*attn_len)

  def forward(self, q, kv):
    self.Q = self.split_heads(torch.matmul(q, self.Wq))
    self.K = self.split_heads(torch.matmul(kv, self.Wk))
    self.V = self.split_heads(torch.matmul(kv, self.Wv))
    dot_prod = torch.matmul(self.Q, self.K.transpose(2,3))
    scaled_dot_prod = dot_prod / math.sqrt(self.dq)
    attn_score = torch.softmax(scaled_dot_prod, dim = -1)
    out = torch.matmul(attn_score, self.V)
    out = self.combine_heads(out)
    out = torch.matmul(out, self.Wo)
    return out


* By default, `mask=None`. Therefore, create and apply the mask while computing the attention scores


In [281]:
import math
class MHMA(nn.Module):
  def __init__(self,dmodel,dq,dk,dv,heads,mask=None):
    super(MHMA,self).__init__()
    # your code goes here
    self.Wq = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(43)))
    self.Wk = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(44)))
    self.Wv = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(45)))
    self.Wo = nn.Parameter(torch.randn(size=(dmodel, dmodel),generator = torch.random.manual_seed(46)))
    self.heads = heads
    self.dq, self.dv = dq, dv
    self.mask = mask

  def split_heads(self, x):
    batch_size, seq_len, embed_dim = x.size()
    return x.view(batch_size, seq_len, self.heads, self.dq).transpose(1,2)

  def combine_heads(self, x):
    batch_size, heads, seq_len, attn_len = x.size()
    return x.transpose(1,2).contiguous().view(batch_size, seq_len, heads*attn_len)

  def forward(self, x):
    # implement forward method
    self.Q = self.split_heads(torch.matmul(x, self.Wq))
    self.K = self.split_heads(torch.matmul(x, self.Wk))
    self.V = self.split_heads(torch.matmul(x, self.Wv))
    dot_prod = torch.matmul(self.Q, self.K.transpose(2,3))
    scaled_dot_prod = dot_prod / math.sqrt(self.dq)
    if self.mask:
       new_mask = torch.triu(scaled_dot_prod, diagonal = 1)
       scaled_dot_prod = scaled_dot_prod.masked_fill(new_mask != 0, -float('inf'))
    attn_score = torch.softmax(scaled_dot_prod, dim = -1)
    out = torch.matmul(attn_score, self.V)
    out = self.combine_heads(out)
    out = torch.matmul(out, self.Wo)
    return out

* Implement the FFN and OutputLayer modules (same as the one you implemented for encoder)

In [228]:
class FFN(nn.Module):
  def __init__(self,dmodel,d_ff):
    super(FFN,self).__init__()
    self.W_1 = nn.Parameter(torch.randn(size=(dmodel,d_ff), generator = torch.random.manual_seed(47))) #Bias not considered
    self.W_2 = nn.Parameter(torch.randn(size=(d_ff,dmodel), generator = torch.random.manual_seed(48))) #Bias not considered
    self.b_1 = nn.Parameter(torch.randn(size=(1,d_ff), generator = torch.random.manual_seed(47)))
    self.b_2 = nn.Parameter(torch.randn(size=(1,dmodel), generator = torch.random.manual_seed(48)))
    self.relu = nn.ReLU()

  def forward(self,x):
    out = self.relu(torch.matmul(x,self.W_1) + self.b_1)
    out = torch.matmul(out,self.W_2) + self.b_2
    return out

In [ ]:
class OutputLayer(nn.Module):

  def __init__(self,dmodel,vocab_size):
    super(OutputLayer,self).__init__()
    self.W_l = nn.Parameter(torch.randn(size=(dmodel,vocab_size), generator = torch.random.manual_seed(49)))
    self.b_l = nn.Parameter(torch.randn(size=(1,vocab_size), generator = torch.random.manual_seed(49)))

  def forward(self,x):
    out = torch.matmul(x, self.W_l) + self.b_l
    return out

* Implement the final decoder layer.

In [229]:
class DecoderLayer(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,d_ff,heads,mask=None):
    super(DecoderLayer,self).__init__()
    self.mhma = MHMA(dmodel,dq,dk,dv,heads,mask=mask)
    self.mhca = MHCA(dmodel,dq,dk,dv,heads)
    self.layer_norm_mhma = torch.nn.LayerNorm(dmodel)
    self.layer_norm_mhca = torch.nn.LayerNorm(dmodel)
    self.layer_norm_ffn = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,enc_rep, x):
    out = self.layer_norm_mhma(self.mhma(x)+x)
    out = self.layer_norm_mhca(self.mhca(out,enc_rep)+out)
    out = self.layer_norm_ffn(self.ffn(out) + out)
    return out

* Create an embedding layer that takes in token_ids and return embeddings for the token_ids

 * Use seed value: 70

In [230]:
class Embed(nn.Module):

  def __init__(self,vocab_size,embed_dim):
    super(Embed,self).__init__()
    w_e = nn.Parameter(torch.randn(size=(vocab_size, embed_dim),generator = torch.random.manual_seed(70)))
    self.embed= nn.Embedding(vocab_size, embed_dim, _weight = w_e)

  def forward(self,x):
    out = self.embed(x.long())
    return out

# Decoder

 * Implement the decoder that has `num_layers` decoder layers

In [231]:
class Decoder(nn.Module):

  def __init__(self,enc_rep,vocab_size,dmodel,dq,dk,dv,d_ff,heads,mask,num_layers=1):
    super(Decoder,self).__init__()
    self.embed_lookup = Embed(vocab_size, dmodel)
    self.dec_layers = DecoderLayer(dmodel,dq,dk,dv,d_ff,heads,mask)
    self.output = OutputLayer(dmodel, vocab_size)

  def forward(self,enc_rep,tar_token_ids):
    out = self.embed_lookup(tar_token_ids)
    out = self.dec_layers(enc_rep, out)
    out = self.output(out)

    return out

* Representation from encoder

 * Since all the decoder layers require the representation from the encoder to compute cross attention, we are going to feed in the random values (Note, it does not require gradient during training)

In [232]:
# do not edit this
enc_rep = torch.randn(size=(batch_size,seq_len,embed_dim),generator=torch.random.manual_seed(10))

# Instantiate the model

In [286]:
model = Decoder(enc_rep,vocab_size,dmodel,dq,dk,dv,d_ff,heads,mask=True)

In [287]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [288]:
def train(enc_rep,tar_token_ids,label_ids,epochs=1000):
  loss_trace = []
  for epoch in range(epochs):
    out = model(enc_rep,tar_token_ids)
    out = out.view(-1,vocab_size)
    target = tar_token_ids.view(-1)
    loss = criterion(out, target.long())
    if ((epoch+1)%100 == 0):
      print(f'Loss in epoch - {epoch+1} is {loss}')
    loss.backward()

    #update parameters
    optimizer.step()
    optimizer.zero_grad()

* Train the model for 1000 epochs

In [289]:
train(enc_rep,token_ids,label_ids,1000)

Loss in epoch - 100 is 1.7969850301742554
Loss in epoch - 200 is 0.9397293329238892
Loss in epoch - 300 is 0.6040338277816772
Loss in epoch - 400 is 0.42025238275527954
Loss in epoch - 500 is 0.30500558018684387
Loss in epoch - 600 is 0.23432469367980957
Loss in epoch - 700 is 0.17738734185695648
Loss in epoch - 800 is 0.14092311263084412
Loss in epoch - 900 is 0.11700345575809479
Loss in epoch - 1000 is 0.09954638034105301


In [290]:
with torch.inference_mode():
  predictions = torch.argmax(model(enc_rep,token_ids),dim=-1)

* The loss will be around 0.17 after 1000 epochs

In [291]:
# number of correct predictions
print(torch.count_nonzero(label_ids==predictions[:,1:]))

tensor(70)


* THe number of correct predictions is close to 66